In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import bs

In [ ]:
path = '../Erdos_Project'

In [11]:
df_pushed_year = pd.read_csv(f'{path}/Modeling/common_NPI_pushed_year_df_with_pca_1_scaled_type_tot_risk_covid.csv').drop('Unnamed: 0',axis=1)
df_origin = pd.read_csv(f'{path}/Modeling/original_year_with_type_risk_tot_risk_pca_covid.csv').drop('Unnamed: 0', axis=1)

In [12]:
df_train = df_pushed_year[df_pushed_year['year']<=2020].copy()
df_val = df_origin[df_origin['year']==2020].copy() # use 2020 data predict for 2021, the df_val_df dataset is in original year
df_val['year']=2021
df_val['Is_Covid']=1
actual_total_y = sum(df_origin[df_origin['year']==2021]['Tot_Mdcr_Pymt_Amt'])
actual_total_y

109259516579.438

In [13]:
set(df_val.columns.tolist())==set(df_train.columns.tolist())

True

In [26]:
RESULT_DF = pd.DataFrame(columns=['formula','val_percentage','train_percentage','val_deviance','pseudo_R^2','offset'])
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,pseudo_R^2,offset


# Null Model

In [15]:
MODEL_null = smf.glm(
    formula="Tot_Mdcr_Pymt_Amt ~ 1",
    data=df_train,
    family=sm.families.Gamma(link=sm.families.links.Log()),
    offset=np.log(df_train["pca_1_scaled"]),
).fit()

# Model A with offset

In [16]:
df_train.columns.tolist()

['Rndrng_NPI',
 'Rndrng_Prvdr_Ent_Cd',
 'Rndrng_Prvdr_State_Abrvtn',
 'Rndrng_Prvdr_Type',
 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind',
 'Tot_HCPCS_Cds',
 'Tot_Benes',
 'Tot_Srvcs',
 'Bene_Avg_Age',
 'Bene_Avg_Risk_Scre',
 'pca_1_scaled',
 'Tot_Risk',
 'APP_Tot_Risk',
 'PrimaryCare_Tot_Risk',
 'MedicalSpecialtyOther_Tot_Risk',
 'LabPathology_Tot_Risk',
 'PharmacyNutrition_Tot_Risk',
 'state',
 'Tot_Mdcr_Pymt_Amt',
 'year',
 'Is_Covid']

In [17]:
df_train['Tot_Mdcr_Pymt_Amt'] = df_train['Tot_Mdcr_Pymt_Amt'].apply(lambda x:x if x!=0 else 1)

In [18]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ Is_Covid
    + cr(year, df=3)
    """

MODEL_1 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    offset = np.log(df_train['pca_1_scaled'])
).fit()

In [19]:
formula.replace('\n','').replace(' ','')[18:].split('+')

['Is_Covid', 'cr(year,df=3)']

In [27]:
y_pred = MODEL_1.predict(df_val,offset=np.log(df_val['pca_1_scaled']))

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_1.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_1.deviance / MODEL_null.deviance

val_d = MODEL_1.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_1.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo_R^2': [r2_dev],
    'offset':['Y']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo_R^2,offset
0,"[Is_Covid, cr(year,df=3)]",0.911559,0.954837,241081.201531,0.000832,Y


In [28]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

/var/folders/46/mjvmwj6x7n5_cdghvmx10bkm0000gn/T/ipykernel_76746/2588865648.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)


,formula,val_percentage,train_percentage,val_deviance,pseudo_R^2,offset
0,"[Is_Covid, cr(year,df=3)]",0.911559,0.954837,241081.201531,0.000832,Y


In [22]:
MODEL_1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:      Tot_Mdcr_Pymt_Amt   No. Observations:              6880944
Model:                            GLM   Df Residuals:                  6880940
Model Family:                   Gamma   Df Model:                            3
Link Function:                    Log   Scale:                          11.265
Method:                          IRLS   Log-Likelihood:            -9.0495e+07
Date:                Sun, 08 Mar 2026   Deviance:                   1.4618e+07
Time:                        10:21:29   Pearson chi2:                 7.75e+07
No. Iterations:                   100   Pseudo R-squ. (CS):          0.0001246
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept          5.394e+06   2.95e+07      0.183      0.855   -5.24e+07    6.32e+07
Is_Covid             -0.1078      0.007    -15.996      0.000      -0.121      -0.095
cr(year, df=3)[0] -5.394e+06   2.95e+07     -0.183      0.855   -6.32e+07    5.24e+07
cr(year, df=3)[1] -5.394e+06   2.95e+07     -0.183      0.855   -6.32e+07    5.24e+07
cr(year, df=3)[2] -5.394e+06   2.95e+07     -0.183      0.855   -6.32e+07    5.24e+07
=====================================================================================
"""

# Model 2 - without offset

In [23]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ Is_Covid
    + cr(year, df=3)
    """

MODEL_2 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

In [29]:
y_pred = MODEL_2.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_2.predict(df_train))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_2.deviance / MODEL_null.deviance

val_d = MODEL_2.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_2.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo_R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo_R^2,offset
0,"[Is_Covid, cr(year,df=3)]",0.964609,0.999948,79703.221835,-0.042779,N


In [30]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,pseudo_R^2,offset
0,"[Is_Covid, cr(year,df=3)]",0.911559,0.954837,241081.201531,0.000832,Y
0,"[Is_Covid, cr(year,df=3)]",0.964609,0.999948,79703.221835,-0.042779,N


In [32]:
MODEL_2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:      Tot_Mdcr_Pymt_Amt   No. Observations:              6880944
Model:                            GLM   Df Residuals:                  6880940
Model Family:                   Gamma   Df Model:                            3
Link Function:                    Log   Scale:                          35.712
Method:                          IRLS   Log-Likelihood:            -9.6993e+07
Date:                Sun, 08 Mar 2026   Deviance:                   1.5256e+07
Time:                        10:27:25   Pearson chi2:                 2.46e+08
No. Iterations:                   100   Pseudo R-squ. (CS):          3.324e-05
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept          5.373e+06   5.25e+07      0.102      0.918   -9.75e+07    1.08e+08
Is_Covid             -0.1067      0.012     -8.894      0.000      -0.130      -0.083
cr(year, df=3)[0] -5.373e+06   5.25e+07     -0.102      0.918   -1.08e+08    9.75e+07
cr(year, df=3)[1] -5.373e+06   5.25e+07     -0.102      0.918   -1.08e+08    9.75e+07
cr(year, df=3)[2] -5.373e+06   5.25e+07     -0.102      0.918   -1.08e+08    9.75e+07
=====================================================================================
"""

In [31]:
RESULT_DF.to_csv(f'{path}/Modeling/GLM_GAMMA_RESULTS.csv')

# Model 3 - year with df=6 instead of 4

In [41]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_3 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    offset = np.log(df_train['pca_1_scaled'])
).fit()

In [45]:
y_pred = MODEL_3.predict(df_val,offset=np.log(df_val['pca_1_scaled']))

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_3.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_3.deviance / MODEL_null.deviance

val_d = MODEL_3.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_3.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev]
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,0.288775


In [ ]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

TypeError: unhashable type: 'list'

In [50]:
y_pred = MODEL_3.predict(df_val,offset=np.log(df_val['pca_1_scaled']))
sum(y_pred)/actual_total_y

0.9148721400280301

In [49]:
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775


# Model 4 - try without offset

In [52]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_4 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

In [94]:
y_pred = MODEL_4.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_4.predict(df_train))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_4.deviance / MODEL_null.deviance

val_d = MODEL_4.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_4.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,[Bene_Avg_Risk_Scre],0.943211,1.006478,403510.537576,0.277827,N


In [60]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N


In [61]:
RESULT_DF.to_csv(f'{path}/Modeling/GLM_GAMMA_RESULTS.csv')

In [62]:
MODEL_4.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:      Tot_Mdcr_Pymt_Amt   No. Observations:              6880944
Model:                            GLM   Df Residuals:                  6880916
Model Family:                   Gamma   Df Model:                           27
Link Function:                    Log   Scale:                          4.9179
Method:                          IRLS   Log-Likelihood:            -8.6479e+07
Date:                Sat, 07 Mar 2026   Deviance:                   1.0652e+07
Time:                        10:30:48   Pearson chi2:                 3.38e+07
No. Iterations:                   100   Pseudo R-squ. (CS):             0.1274
Covariance Type:            nonrobust                                         
==================================================================================================================================================
                                                                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                          9.8896      0.004   2701.754      0.000       9.882       9.897
C(Rndrng_Prvdr_Type)[T.AcuteCare]                                                  0.6999      0.004    157.644      0.000       0.691       0.709
C(Rndrng_Prvdr_Type)[T.Anesthesia]                                                 0.1694      0.004     45.889      0.000       0.162       0.177
C(Rndrng_Prvdr_Type)[T.BehavioralHealth]                                           0.2023      0.004     46.163      0.000       0.194       0.211
C(Rndrng_Prvdr_Type)[T.CardioVascularSurgery]                                      1.9100      0.011    179.295      0.000       1.889       1.931
C(Rndrng_Prvdr_Type)[T.Cardiology]                                                 2.0628      0.006    354.719      0.000       2.051       2.074
C(Rndrng_Prvdr_Type)[T.FacilitySupplierProgram]                                    1.8412      0.004    439.876      0.000       1.833       1.849
C(Rndrng_Prvdr_Type)[T.LabPathology]                                               2.7478      0.007    366.545      0.000       2.733       2.762
C(Rndrng_Prvdr_Type)[T.MedicalSpecialtyOther]                                      1.7638      0.004    472.908      0.000       1.757       1.771
C(Rndrng_Prvdr_Type)[T.OBGYN]                                                     -0.3328      0.006    -58.053      0.000      -0.344      -0.322
C(Rndrng_Prvdr_Type)[T.OncologyHeme]                                               2.8714      0.007    430.190      0.000       2.858       2.885
C(Rndrng_Prvdr_Type)[T.PharmacyNutrition]                                          0.4901      0.020     24.608      0.000       0.451       0.529
C(Rndrng_Prvdr_Type)[T.PrimaryCare]                                                1.0728      0.003    365.441      0.000       1.067       1.079
C(Rndrng_Prvdr_Type)[T.RadiologyImaging]                                           7.7755      0.838      9.276      0.000       6.133       9.418
C(Rndrng_Prvdr_Type)[T.RehabTherapy]                                               0.5507      0.004    152.906      0.000       0.544       0.558
C(Rndrng_Prvdr_Type)[T.SurgeryOther]                                               1.4852      0.004    412.490      0.000       1.478       1.492
C(Rndrng_Prvdr_Type)[T.UnknownOther]                                               1.4600      0.051     28.849      0.000       1.361       1.559
C(Rndrng_Prvdr_Type)[T.VisionHearing]                                              1.7924      0.004    411.385      0.000       1.784       1.

# Model 5 - year df=7 instead of 6

In [63]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=7, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_5 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

In [95]:
y_pred = MODEL_5.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_5.predict(df_train))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_5.deviance / MODEL_null.deviance

val_d = MODEL_5.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_5.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,[Bene_Avg_Risk_Scre],0.942448,1.006254,403381.998303,0.277829,N


In [65]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N


In [66]:
RESULT_DF.to_csv(f'{path}/Modeling/GLM_GAMMA_RESULTS.csv')

# Model 6 - without offset without type_risk label

In [67]:
df_train.columns

Index(['Rndrng_NPI', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_State_Abrvtn',
       'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'Tot_HCPCS_Cds',
       'Bene_Avg_Age', 'Bene_Avg_Risk_Scre', 'pca_1_scaled',
       'FacilitySupplierProgram_Risk', 'RadiologyImaging_Risk',
       'LabPathology_Risk', 'Tot_Mdcr_Pymt_Amt', 'year', 'Is_Covid', 'state'],
      dtype='object')

In [68]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type)
    + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_6 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

In [70]:
y_pred = MODEL_6.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_6.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_6.deviance / MODEL_null.deviance

val_d = MODEL_6.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_6.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.01165,329740.057022,0.274232,N


In [71]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N


# Model 7 - without offset without type_risk label

In [72]:
df_train.columns

Index(['Rndrng_NPI', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_State_Abrvtn',
       'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'Tot_HCPCS_Cds',
       'Bene_Avg_Age', 'Bene_Avg_Risk_Scre', 'pca_1_scaled',
       'FacilitySupplierProgram_Risk', 'RadiologyImaging_Risk',
       'LabPathology_Risk', 'Tot_Mdcr_Pymt_Amt', 'year', 'Is_Covid', 'state'],
      dtype='object')

In [74]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    + Bene_Avg_Age
    """

MODEL_7 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

y_pred = MODEL_7.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_7.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_7.deviance / MODEL_null.deviance

val_d = MODEL_7.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_7.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,0.310523,N


In [75]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,NaN,0.310523,NaN,N


In [76]:
RESULT_DF.to_csv(f'{path}/Modeling/GLM_GAMMA_RESULTS.csv')

In [77]:
MODEL_7.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:      Tot_Mdcr_Pymt_Amt   No. Observations:              6880944
Model:                            GLM   Df Residuals:                  6880915
Model Family:                   Gamma   Df Model:                           28
Link Function:                    Log   Scale:                          6.3748
Method:                          IRLS   Log-Likelihood:            -8.7531e+07
Date:                Sat, 07 Mar 2026   Deviance:                   1.0170e+07
Time:                        11:25:24   Pearson chi2:                 3.83e+07
No. Iterations:                   100   Pseudo R-squ. (CS):             0.1096
Covariance Type:            nonrobust                                         
==================================================================================================================================================
                                                                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                          6.2097      0.014    457.232      0.000       6.183       6.236
C(Rndrng_Prvdr_Type)[T.AcuteCare]                                                  0.6873      0.005    135.886      0.000       0.677       0.697
C(Rndrng_Prvdr_Type)[T.Anesthesia]                                                 0.1853      0.004     44.070      0.000       0.177       0.194
C(Rndrng_Prvdr_Type)[T.BehavioralHealth]                                           0.6754      0.005    129.286      0.000       0.665       0.686
C(Rndrng_Prvdr_Type)[T.CardioVascularSurgery]                                      1.8092      0.012    149.017      0.000       1.785       1.833
C(Rndrng_Prvdr_Type)[T.Cardiology]                                                 1.8548      0.007    278.234      0.000       1.842       1.868
C(Rndrng_Prvdr_Type)[T.FacilitySupplierProgram]                                    1.7542      0.005    365.329      0.000       1.745       1.764
C(Rndrng_Prvdr_Type)[T.LabPathology]                                               2.7451      0.009    321.575      0.000       2.728       2.762
C(Rndrng_Prvdr_Type)[T.MedicalSpecialtyOther]                                      1.7061      0.004    401.523      0.000       1.698       1.714
C(Rndrng_Prvdr_Type)[T.OBGYN]                                                     -0.1865      0.007    -28.303      0.000      -0.199      -0.174
C(Rndrng_Prvdr_Type)[T.OncologyHeme]                                               2.7391      0.008    359.780      0.000       2.724       2.754
C(Rndrng_Prvdr_Type)[T.PharmacyNutrition]                                          1.2781      0.023     56.362      0.000       1.234       1.323
C(Rndrng_Prvdr_Type)[T.PrimaryCare]                                                0.9551      0.003    284.022      0.000       0.948       0.962
C(Rndrng_Prvdr_Type)[T.RadiologyImaging]                                           7.2420      0.954      7.589      0.000       5.372       9.112
C(Rndrng_Prvdr_Type)[T.RehabTherapy]                                               0.4671      0.004    113.413      0.000       0.459       0.475
C(Rndrng_Prvdr_Type)[T.SurgeryOther]                                               1.3694      0.004    331.993      0.000       1.361       1.377
C(Rndrng_Prvdr_Type)[T.UnknownOther]                                               1.3870      0.058     24.072      0.000       1.274       1.500
C(Rndrng_Prvdr_Type)[T.VisionHearing]                                              1.5474      0.005    309.271      0.000       1.538       1.

# Model 8 - with state

In [78]:
df_train.columns

Index(['Rndrng_NPI', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_State_Abrvtn',
       'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'Tot_HCPCS_Cds',
       'Bene_Avg_Age', 'Bene_Avg_Risk_Scre', 'pca_1_scaled',
       'FacilitySupplierProgram_Risk', 'RadiologyImaging_Risk',
       'LabPathology_Risk', 'Tot_Mdcr_Pymt_Amt', 'year', 'Is_Covid', 'state'],
      dtype='object')

In [80]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type) + C(state)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_8 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

y_pred = MODEL_8.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_8.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_8.deviance / MODEL_null.deviance

val_d = MODEL_8.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_8.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[C(Rndrng_Prvdr_Type), C(state), FacilitySuppl...",0.921437,1.044356,403821.662431,0.288531,N


In [81]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,NaN,0.310523,NaN,N


# Model 9

In [82]:
df_train.columns

Index(['Rndrng_NPI', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_State_Abrvtn',
       'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'Tot_HCPCS_Cds',
       'Bene_Avg_Age', 'Bene_Avg_Risk_Scre', 'pca_1_scaled',
       'FacilitySupplierProgram_Risk', 'RadiologyImaging_Risk',
       'LabPathology_Risk', 'Tot_Mdcr_Pymt_Amt', 'year', 'Is_Covid', 'state'],
      dtype='object')

In [85]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ C(Rndrng_Prvdr_Type) + C(Rndrng_Prvdr_Mdcr_Prtcptg_Ind)
    + FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_9 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

y_pred = MODEL_9.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_9.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_9.deviance / MODEL_null.deviance

val_d = MODEL_9.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_9.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[C(Rndrng_Prvdr_Type), C(Rndrng_Prvdr_Mdcr_Prt...",0.925326,1.045257,401870.041176,0.278306,N


In [86]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,NaN,0.310523,NaN,N


In [87]:
MODEL_9.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:      Tot_Mdcr_Pymt_Amt   No. Observations:              6880944
Model:                            GLM   Df Residuals:                  6880915
Model Family:                   Gamma   Df Model:                           28
Link Function:                    Log   Scale:                          4.9283
Method:                          IRLS   Log-Likelihood:            -8.6487e+07
Date:                Sat, 07 Mar 2026   Deviance:                   1.0645e+07
Time:                        12:14:16   Pearson chi2:                 3.39e+07
No. Iterations:                   100   Pseudo R-squ. (CS):             0.1273
Covariance Type:            nonrobust                                         
==================================================================================================================================================
                                                                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                          8.4991      0.031    272.067      0.000       8.438       8.560
C(Rndrng_Prvdr_Type)[T.AcuteCare]                                                  0.6995      0.004    157.383      0.000       0.691       0.708
C(Rndrng_Prvdr_Type)[T.Anesthesia]                                                 0.1681      0.004     45.472      0.000       0.161       0.175
C(Rndrng_Prvdr_Type)[T.BehavioralHealth]                                           0.2028      0.004     46.209      0.000       0.194       0.211
C(Rndrng_Prvdr_Type)[T.CardioVascularSurgery]                                      1.9093      0.011    179.043      0.000       1.888       1.930
C(Rndrng_Prvdr_Type)[T.Cardiology]                                                 2.0618      0.006    354.163      0.000       2.050       2.073
C(Rndrng_Prvdr_Type)[T.FacilitySupplierProgram]                                    1.8403      0.004    439.177      0.000       1.832       1.848
C(Rndrng_Prvdr_Type)[T.LabPathology]                                               2.7468      0.008    366.025      0.000       2.732       2.761
C(Rndrng_Prvdr_Type)[T.MedicalSpecialtyOther]                                      1.7630      0.004    472.172      0.000       1.756       1.770
C(Rndrng_Prvdr_Type)[T.OBGYN]                                                     -0.3343      0.006    -58.258      0.000      -0.346      -0.323
C(Rndrng_Prvdr_Type)[T.OncologyHeme]                                               2.8711      0.007    429.679      0.000       2.858       2.884
C(Rndrng_Prvdr_Type)[T.PharmacyNutrition]                                          0.4963      0.020     24.891      0.000       0.457       0.535
C(Rndrng_Prvdr_Type)[T.PrimaryCare]                                                1.0714      0.003    364.565      0.000       1.066       1.077
C(Rndrng_Prvdr_Type)[T.RadiologyImaging]                                           7.7741      0.839      9.265      0.000       6.129       9.419
C(Rndrng_Prvdr_Type)[T.RehabTherapy]                                               0.5543      0.004    153.474      0.000       0.547       0.561
C(Rndrng_Prvdr_Type)[T.SurgeryOther]                                               1.4839      0.004    411.681      0.000       1.477       1.491
C(Rndrng_Prvdr_Type)[T.UnknownOther]                                               1.4585      0.051     28.789      0.000       1.359       1.558
C(Rndrng_Prvdr_Type)[T.VisionHearing]                                              1.7911      0.004    410.641      0.000       1.783       1.

# Model 10 - without provider

In [88]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ FacilitySupplierProgram_Risk + RadiologyImaging_Risk + LabPathology_Risk + Is_Covid
    + bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_10 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

y_pred = MODEL_10.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_10.predict(df_train,offset=np.log(df_train['pca_1_scaled'])))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_10.deviance / MODEL_null.deviance

val_d = MODEL_10.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_10.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[FacilitySupplierProgram_Risk, RadiologyImagin...",0.976153,1.04539,108207.790976,-0.0044,N


In [89]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,NaN,0.310523,NaN,N


# Model 11

In [ ]:
formula = """
    Tot_Mdcr_Pymt_Amt
    ~ bs(year, df=6, include_intercept=False, lower_bound=2014, upper_bound=2023)
    + Bene_Avg_Risk_Scre
    """

MODEL_11 = smf.glm(
    formula = formula,
    data = df_train,
    family = sm.families.Gamma(link=sm.families.links.Log()),
    #offset = np.log(df_train['pca_1_scaled'])
).fit()

y_pred = MODEL_11.predict(df_val)

val_p = sum(y_pred)/actual_total_y

train_p = sum(MODEL_11.predict(df_train))/sum(df_train['Tot_Mdcr_Pymt_Amt'])

r2_dev = 1 - MODEL_11.deviance / MODEL_null.deviance

val_d = MODEL_11.model.family.deviance(df_val['Tot_Mdcr_Pymt_Amt'],y_pred,scale=MODEL_11.scale)

temp = pd.DataFrame({
    'formula': [formula.replace('\n','').replace(' ','')[18:].split('+')],
    'val_percentage': [val_p],
    'train_percentage': [train_p],
    'val_deviance': [val_d],
    'pseudo-R^2': [r2_dev],
    'offset':['N']
})
temp

,formula,val_percentage,train_percentage,val_deviance,pseudo-R^2,offset
0,"[bs(year,df=6,include_intercept=False,lower_bo...",0.980066,1.012014,68043.064045,-0.016786,N


In [97]:
RESULT_DF = pd.concat([RESULT_DF,temp],axis=0)
RESULT_DF

,formula,val_percentage,train_percentage,val_deviance,psudo-R^2,pseudo-R^2,without offset,offset
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.885083,0.973222,473374.117165,NaN,0.281022,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.897992,0.978895,485465.387123,NaN,0.288803,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.914872,0.978754,487309.643266,NaN,0.288775,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,Y,NaN
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.943211,1.045239,403510.537576,NaN,0.277827,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.942448,1.045005,403381.998303,NaN,0.277829,NaN,N
0,"[C(Rndrng_Prvdr_Type), Is_Covid, bs(year,df=6,...",0.925562,1.011650,329740.057022,NaN,0.274232,NaN,N
0,"[C(Rndrng_Prvdr_Type), FacilitySupplierProgram...",0.909448,1.038956,295535.030813,NaN,0.310523,NaN,N


In [98]:
RESULT_DF.to_csv(f'{path}/Modeling/GLM_GAMMA_RESULTS.csv')